# Loremaster Retrieval Evaluation

This notebook evaluates and compares five Elasticsearch query strategies against a benchmark dataset of 150 WoW queries.

## Strategies

| Strategy | Description |
|---|---|
| `original` | Single `best_fields` query with fuzziness and 75% `minimum_should_match`. The original production query. |
| `cross_fields` | Single `cross_fields` query. Treats all terms as belonging to one combined field across title, summary, and content. |
| `best_fields` | Single `best_fields` query with fuzziness, no `minimum_should_match`. |
| `rrf_bm25` | RRF combining `cross_fields` and `best_fields` retrievers. No ELSER. |
| `rrf_elser` | RRF combining `cross_fields`, `best_fields`, and ELSER sparse vector retrievers. Current production query. |

## Metrics

- **Hit@5** — did the expected article appear in the top 5 results?
- **MRR (Mean Reciprocal Rank)** — how highly was the expected article ranked? Score of 1.0 means it was always the top result.

## Benchmark Dataset

150 queries across five categories (lore, quest, item, gameplay, long_tail) and two query types (conversational and short). See `evals/data/benchmark.json`.

In [ ]:
import sys, json
from pathlib import Path
sys.path.append(str(Path('../..').resolve()))

from elasticsearch import Elasticsearch
from config import ES_ENDPOINT, ES_API_KEY, ES_INDEX
from evals.retrieval.strategies import STRATEGIES
from evals.retrieval.metrics import hit_rate, mrr

es    = Elasticsearch(ES_ENDPOINT, api_key=ES_API_KEY)
TOP_K = 5

## Load Benchmark

Loads the benchmark dataset and prints a summary of query counts by type.

In [ ]:
benchmark  = json.loads(Path('../../evals/data/benchmark.json').read_text())
queries    = [b['query']    for b in benchmark]
expected   = [b['expected'] for b in benchmark]
categories = [b['category'] for b in benchmark]
types      = [b.get('type', 'conversational') for b in benchmark]

print(f'Total queries:     {len(benchmark)}')
print(f'Conversational:    {sum(1 for t in types if t == "conversational")}')
print(f'Short:             {sum(1 for t in types if t == "short")}')

## Run All Strategies

Runs each strategy against all 150 queries and caches the results. This avoids re-querying Elasticsearch for every breakdown section below.

In [ ]:
def search(query_body):
    result = es.search(index=ES_INDEX, body={**query_body, 'size': TOP_K, '_source': ['title']})
    return [h['_source']['title'] for h in result['hits']['hits']]

cached = {}
for name, strategy in STRATEGIES.items():
    print(f'Running {name}...')
    cached[name] = [search(strategy['query'](q)) for q in queries]
print('Done.')

## Overall Results

Hit@5 and MRR across all 150 queries.

In [ ]:
print(f"{'Strategy':<20} {'Hit@5':>8}   {'MRR':>8}")
print('-' * 45)
for name in STRATEGIES:
    hr        = hit_rate(cached[name], expected, k=TOP_K)
    mrr_score = mrr(cached[name], expected, k=TOP_K)
    print(f'{name:<20} {hr:>8.2%}   {mrr_score:>8.3f}')

## Per-Category Breakdown

Results broken down by query category (lore, quest, item, gameplay, long_tail).

In [ ]:
for category in sorted(set(categories)):
    indices = [i for i, c in enumerate(categories) if c == category]
    print(f'\n{category} ({len(indices)} queries)')
    for name in STRATEGIES:
        results   = [cached[name][i] for i in indices]
        exp       = [expected[i] for i in indices]
        hr        = hit_rate(results, exp, k=TOP_K)
        mrr_score = mrr(results, exp, k=TOP_K)
        print(f'  {name:<20} {hr:>8.2%}   MRR {mrr_score:.3f}')

## By Query Type

Conversational queries (e.g. "Hey, where do I find...") vs short direct queries (e.g. "Arthas death knight lore"). This breakdown shows how each strategy handles different query styles.

In [ ]:
for qtype in sorted(set(types)):
    indices = [i for i, t in enumerate(types) if t == qtype]
    print(f'\n{qtype} ({len(indices)} queries)')
    for name in STRATEGIES:
        results   = [cached[name][i] for i in indices]
        exp       = [expected[i] for i in indices]
        hr        = hit_rate(results, exp, k=TOP_K)
        mrr_score = mrr(results, exp, k=TOP_K)
        print(f'  {name:<20} {hr:>8.2%}   MRR {mrr_score:.3f}')